# Scheduling-context features for XGBoost

This standalone notebook starts from the **original preprocessed CSVs**. It retains the earlier full feature set and adds relative price-versus-water-value features. Price-block and near-term opportunity features remain available as optional experiments. It writes `hourly_train.csv`, `hourly_test.csv`, `cases.csv`, and `meta.json` to **`engineered_scheduling/`**. The original files and other feature-engineering notebooks are untouched.

In Colab, upload the original four files together, then run all cells on CPU. Only NumPy and pandas are needed. The feature logic never reads labels or fits statistics across cases. Observed future-storage changes (`dvol_*`) are excluded.

**Default: 262 features = 169 original safe features + 84 earlier additions + 9 relative-spread features.** The defaults follow the fixed-parameter validation comparison below; block context and local opportunity are off.

## What is new?

| New group | Columns | Scheduling information |
|---|---:|---|
| Positive block spread sum | 9, one per plant | Total `price − upstream water value` over the uninterrupted above-water-value block containing the current hour; zero otherwise |
| Unfavorable block duration | 9 | Length of the uninterrupted block where price is at or below upstream water value; zero during favorable blocks |
| Relative price spread | 9 | `(price − upstream water value) / (abs(price) + abs(upstream water value) + 1)`; a bounded, dimensionless comparison across price levels |
| Near-term positive spread | 9 | Mean of `max(price − upstream water value, 0)` over the current and next five hours, shortened at the horizon end |

These complement existing positive-block length and remaining-week averages: they distinguish a strong price block from a weak one, a short unfavorable gap from a long one, and nearby opportunities from distant ones. **They are economic/continuity proxies, not exact profits, startup costs, or operating rules.** There is no generator-state history in the preprocessed inputs, so none is invented.

The flags below select the candidate groups. The existing full feature set has **253 features**; block context adds 18, local opportunity adds 9, and relative spread adds 9. Disabling all three reproduces the earlier 253-feature set; enabling all three gives 289. The relative-spread denominator uses a **1 EUR/MWh stabilizer** and stays positive even for zero or negative prices. More features do not guarantee a higher validation or Kaggle score.

The retained 84 features cover price windows, upstream/net water-value comparisons, positive-block duration, remaining-week opportunities, aggregate inflow, initial-storage constraint margins, plant fill summaries, and weekly inflow relative to initial storage. All hourly operations stay within each 168-hour case. Price windows include the current hour; cumulative inflow before an hour excludes that hour, while remaining inflow includes it. Inflow conversion is `3600 / 1e6` per hour, giving million m³.

The project PDFs define full-week prices/inflows and terminal water values as scheduling inputs; looking ahead within those supplied inputs is appropriate. Initial-storage margins are not future reservoir trajectories. The existing CSVs retain hourly inflow only as a total, so exact hourly reservoir inflows cannot be recovered here. Fill summaries are unweighted and unclipped; reservoirs missing from the preprocessed storage table are not assigned invented storage values.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Config: set INPUT_DIR explicitly if the four original files are elsewhere.
INPUT_DIR = None
HORIZON = 168
ADD_BLOCK_CONTEXT = False       # 18 columns: block magnitude and unfavorable-block duration.
ADD_LOCAL_OPPORTUNITY = False   # 9 columns: positive spread in the next six hours.
ADD_RELATIVE_SPREAD = True    # 9 columns: dimensionless price vs. upstream water value.
FILES = ("hourly_train.csv", "hourly_test.csv", "cases.csv", "meta.json")

if INPUT_DIR is None:
    INPUT_DIR = next((p for p in (Path("."), Path("processed"), Path("../processed"))
                      if all((p / name).is_file() for name in FILES)), None)
if INPUT_DIR is None:
    raise FileNotFoundError("Upload all four preprocessed files together, or set INPUT_DIR.")
INPUT_DIR = Path(INPUT_DIR).resolve()
OUTPUT_DIR = (INPUT_DIR.parent if INPUT_DIR.name == "processed" else INPUT_DIR) / "engineered_scheduling"

meta = json.loads((INPUT_DIR / "meta.json").read_text())
if "feature_engineering" in meta:
    raise ValueError("Start from the original preprocessed files, not an engineered dataset.")
cases = pd.read_csv(INPUT_DIR / "cases.csv")
if cases["Run No"].isna().any() or cases["Run No"].duplicated().any():
    raise ValueError("cases.csv needs one non-missing Run No per case.")
base_hourly = [c for c in meta["hourly_feature_cols"] if not c.startswith("dvol_")]
base_static = [c for c in meta["static_feature_cols"] if not c.startswith("dvol_")]
removed = [c for c in cases if c.startswith("dvol_")]
print(f"Input: {INPUT_DIR}\nOutput: {OUTPUT_DIR}")

Input: /Users/mjosefsen/Documents/NTNU/TDT4173/github/processed
Output: /Users/mjosefsen/Documents/NTNU/TDT4173/github/engineered_scheduling


## Feature calculations

Three small helpers calculate window means, remaining sums and totals within continuous blocks. The case-level and hourly functions preserve the original identifiers and labels. Detailed numerical tests are kept outside the notebook; the code uses only essential input checks.

In [2]:
# -----------------------------
# Feature calculations (pure functions; never read target columns)
# -----------------------------
def window_mean(a, width, forward):
    """Case × hour array; include the current hour and truncate at either edge."""
    a = np.asarray(a, dtype=np.float64)
    h = a.shape[1]
    t = np.arange(h)
    left = t if forward else np.maximum(0, t - width + 1)
    right = np.minimum(h, t + width) if forward else t + 1
    cumulative = np.pad(np.cumsum(a, axis=1), ((0, 0), (1, 0)))
    return (cumulative[:, right] - cumulative[:, left]) / (right - left)


def remaining_sum(a):
    """Sum from the current hour through the last hour, within each case."""
    return np.cumsum(np.asarray(a, dtype=np.float64)[:, ::-1], axis=1)[:, ::-1]


def block_total(values, active):
    """Sum values over the active block containing each hour; zero outside it."""
    left = np.zeros_like(values, dtype=float)
    right = np.zeros_like(left)
    for t in range(values.shape[1]):
        previous = left[:, t - 1] if t else 0
        left[:, t] = np.where(active[:, t], previous + values[:, t], 0)
    for t in range(values.shape[1] - 1, -1, -1):
        following = right[:, t + 1] if t + 1 < values.shape[1] else 0
        right[:, t] = np.where(active[:, t], following + values[:, t], 0)
    return left + right - np.where(active, values, 0)


def engineer_cases(source_cases, metadata):
    result = source_cases.drop(columns=[c for c in source_cases if c.startswith("dvol_")]).copy()
    new = {}
    for plant, mapping in metadata["plant_map"].items():
        side_fill = {}
        for side in ("up", "down"):
            columns = [f"fill_{r}" for r in mapping[side]]
            if columns and all(c in result for c in columns):
                side_fill[side] = result[columns].mean(axis=1).to_numpy()
                new[f"fe_initial_fill_{side}_{plant}"] = side_fill[side]
        if "up" in side_fill and "down" in side_fill:
            new[f"fe_initial_fill_up_minus_down_{plant}"] = side_fill["up"] - side_fill["down"]

    for column in source_cases:
        if not column.startswith("vol_"):
            continue
        reservoir = column[len("vol_"):]
        inflow_column = f"inflow_{reservoir}"
        if inflow_column not in result:
            continue
        initial = result[column].to_numpy(dtype=float)
        incoming = result[inflow_column].to_numpy(dtype=float) * HORIZON * 3600 / 1e6
        if (initial < 0).any() or (incoming < 0).any():
            raise ValueError("Inflow-share features require nonnegative storage and inflow.")
        denominator = initial + incoming
        new[f"fe_inflow_share_of_initial_plus_week_{reservoir}"] = np.divide(
            incoming, denominator, out=np.zeros_like(incoming), where=denominator > 0)
    additions = pd.DataFrame(new, index=result.index, dtype=np.float32)
    return pd.concat([result, additions], axis=1), list(additions.columns)


def engineer_hourly(source_hourly, source_cases, metadata):
    # Keep all window and block calculations inside each case.
    result = source_hourly.drop(columns=[c for c in source_hourly if c.startswith("dvol_")])
    result = result.sort_values(["Run No", "t"]).reset_index(drop=True)
    ids = result["Run No"].drop_duplicates().to_numpy()
    expected_t = np.tile(np.arange(HORIZON), len(ids))
    if (result["Run No"].isna().any() or not len(ids)
            or not result.groupby("Run No").size().eq(HORIZON).all()
            or not np.array_equal(result["t"].to_numpy(), expected_t)):
        raise ValueError("Each case must have one non-missing Run No and exactly t=0..167.")
    aligned = source_cases.set_index("Run No").loc[ids]
    shape = (len(ids), HORIZON)
    price = result["price"].to_numpy(dtype=float).reshape(shape)
    count_remaining = HORIZON - np.arange(HORIZON)
    new = {}
    for width in (6, 24):
        for direction, forward in (("past", False), ("next", True)):
            new[f"fe_price_minus_{direction}{width}h_mean"] = price - window_mean(price, width, forward)
    new["fe_remaining_price_max_minus_current"] = np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price
    new["fe_remaining_price_mean_minus_current"] = remaining_sum(price) / count_remaining - price

    extra = {}
    for plant in metadata["plant_map"]:
        spread = price - aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
        above = spread > 0
        new[f"fe_price_minus_wvnet_{plant}"] = price - aligned[f"wvnet_{plant}"].to_numpy(dtype=float)[:, None]
        new[f"fe_above_wv_block_hours_{plant}"] = block_total(np.ones_like(spread), above)
        new[f"fe_remaining_above_wv_fraction_{plant}"] = remaining_sum(above) / count_remaining
        new[f"fe_remaining_positive_spread_mean_{plant}"] = remaining_sum(np.maximum(spread, 0)) / count_remaining
        if ADD_BLOCK_CONTEXT:
            extra[f"fe_positive_block_spread_sum_{plant}"] = block_total(spread, above)
            extra[f"fe_below_wv_block_hours_{plant}"] = block_total(np.ones_like(spread), ~above)
        if ADD_LOCAL_OPPORTUNITY:
            extra[f"fe_next6h_positive_spread_mean_{plant}"] = window_mean(np.maximum(spread, 0), 6, True)
        if ADD_RELATIVE_SPREAD:
            water_value = aligned[f"wvup_{plant}"].to_numpy(dtype=float)[:, None]
            extra[f"fe_relative_price_spread_{plant}"] = spread / (np.abs(price) + np.abs(water_value) + 1.0)

    inflow = result["inflow_total_h"].to_numpy(dtype=float).reshape(shape)
    volume = inflow * 3600 / 1e6
    new["fe_inflow_before_hour_mm3"] = np.pad(np.cumsum(volume, axis=1)[:, :-1], ((0, 0), (1, 0)))
    new["fe_inflow_remaining_mm3"] = remaining_sum(volume)
    new["fe_inflow_minus_week_mean"] = inflow - inflow.mean(axis=1, keepdims=True)
    for column in result:
        if column.startswith("minvol_h_"):
            reservoir = column[len("minvol_h_"):]
            initial = aligned[f"vol_{reservoir}"].to_numpy(dtype=float)[:, None]
            new[f"fe_initial_volume_minus_minvol_{reservoir}"] = initial - result[column].to_numpy(dtype=float).reshape(shape)

    new.update(extra)
    additions = pd.DataFrame({name: a.ravel().astype(np.float32) for name, a in new.items()})
    return pd.concat([result, additions], axis=1), list(additions.columns)

In [3]:
# Build both datasets. Only the dataset and its metadata are exported.
engineered_cases, added_static = engineer_cases(cases, meta)
if not np.isfinite(engineered_cases[base_static + added_static].to_numpy(dtype=float)).all():
    raise ValueError("Missing/infinite case features; check cases.csv.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
added_hourly = None
for split in ("train", "test"):
    filename = f"hourly_{split}.csv"
    hourly = pd.read_csv(INPUT_DIR / filename)
    targets = meta["y_cols"] if split == "train" else []
    expected_columns = ["Run No"] + meta["hourly_feature_cols"] + targets
    expected_runs = set(cases.loc[cases["split"].eq(split), "Run No"])
    if list(hourly.columns) != expected_columns or set(hourly["Run No"]) != expected_runs:
        raise ValueError(f"{filename} does not match cases.csv / meta.json.")

    enriched, names = engineer_hourly(hourly, cases, meta)
    if added_hourly is not None and names != added_hourly:
        raise ValueError("Train and test feature columns differ.")
    added_hourly = names
    if not np.isfinite(enriched[base_hourly + names].to_numpy()).all():
        raise ValueError(f"Missing/infinite features in {filename}; check the input files.")
    enriched[["Run No"] + base_hourly + names + targets].to_csv(
        OUTPUT_DIR / filename, index=False, float_format="%.12g", chunksize=10000)
    print(f"Saved {filename}: {len(enriched):,} rows")
    del hourly, enriched

meta["hourly_feature_cols"] = base_hourly + added_hourly
meta["static_feature_cols"] = base_static + added_static
meta["use_end_of_week_volume"] = False
meta["feature_engineering"] = {
    "version": 3, "variant": "scheduling_context", "added_hourly_features": added_hourly,
    "added_static_features": added_static, "removed_features": removed,
    "horizon": HORIZON, "windows_include_current_hour": True, "uses_labels": False,
    "enabled_groups": {"block_context": ADD_BLOCK_CONTEXT, "local_opportunity": ADD_LOCAL_OPPORTUNITY, "relative_spread": ADD_RELATIVE_SPREAD},
    "feature_groups": {
        "block_context": [c for c in added_hourly if c.startswith(("fe_positive_block_spread_sum_", "fe_below_wv_block_hours_"))],
        "local_opportunity": [c for c in added_hourly if c.startswith("fe_next6h_positive_spread_mean_")],
        "relative_spread": [c for c in added_hourly if c.startswith("fe_relative_price_spread_")],
    },
}
engineered_cases.to_csv(OUTPUT_DIR / "cases.csv", index=False, float_format="%.12g")
(OUTPUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
total = len(meta["hourly_feature_cols"]) + len(meta["static_feature_cols"])
print(f"Done: {len(added_hourly)} hourly + {len(added_static)} static additions; {total} total features.")

Saved hourly_train.csv: 490,896 rows
Saved hourly_test.csv: 121,800 rows
Done: 57 hourly + 36 static additions; 262 total features.


## Fixed-parameter validation check

The default feature group was selected after a matched **2022 development-validation** comparison. Each run used 2,551 training cases, 365 validation cases, and the same six-case overlap exclusion. These are local **CPU / XGBoost 3.4.1** results, not Kaggle scores; the CPU baseline was rerun rather than compared against a different GPU run.

| Feature set | Total features | Micro ROC-AUC |
|---|---:|---:|
| Earlier full feature set | 253 | 0.98266 |
| Full + block context + six-hour opportunity | 280 | 0.98247 |
| **Full + relative price spread (default)** | **262** | **0.98733** |

All runs used 300 trees, learning rate 0.05, depth 5, minimum child weight 5, row/column sampling 0.8, L2 regularization 1, seed 1337 and four CPU threads. Adding relative spreads improved all 14 per-unit AUCs in this comparison. For example, Lio improved from 0.91677 to 0.95242.

The 27 block/context additions lowered the overall score slightly, so they are disabled by default. The nine relative spreads increased it by **0.00467**. This is a single-year, single-seed development comparison used to choose features, not an untouched-test estimate. Check other chronological folds and Kaggle before assuming the gain generalizes. Detailed training runs were performed separately to keep this notebook focused on feature engineering; no model or report files are exported here.

## Use the new data

After all cells finish, replace the entire automatic path assignment in your **training** notebook with:

```python
PROCESSED_PATH = Path("/content/engineered_scheduling")  # Colab
```

Locally, use `Path("../engineered_scheduling")` from `github/notebooks/`. Use the generated `meta.json` and all CSVs from this same folder, including `hourly_test.csv` for submission predictions. Rerun loading and training to replace old arrays/models. Your metadata-based `FEATS` list discovers all enabled features automatically.

Compare against the existing full-feature dataset with the **same XGBoost settings, device, seed and validation split**, retaining the gap between overlapping weekly cases. Start with your original parameters, rather than changing features and tuning together. To test a group separately, change the flags and rerun this notebook; this replaces only `engineered_scheduling/`.

The separate hyperparameter-search notebook was deliberately restricted to the original dataset. Its guard will reject these files; use your regular training notebook for this comparison. Validation results guide which features to retain, but cannot guarantee a public or private leaderboard gain.